# FIRST SOLUTIONSOLUTION 

In [87]:
# Import necessary libraries
import asyncio
import httpx
import json
import pandas as pd
from datetime import datetime
import random
import time
import nest_asyncio
from bs4 import BeautifulSoup
import os
from dotenv import load_dotenv
from typing import Optional, Dict, Any, List

# Enable nested asyncio for Jupyter
nest_asyncio.apply()

# Load environment variables if any
load_dotenv()

False

In [89]:
async def discover_api_endpoint(hotel_code: str, check_in: str, check_out: str, adults: int = 2) -> str:
    """
    Improved API endpoint discovery with better headers, delays, and error handling
    Returns a known working endpoint if discovery fails
    """
    # Known working endpoints from manual inspection
    KNOWN_ENDPOINTS = [
        "https://www.expedia.com/m/api/hotel/offers",
        "https://www.expedia.com/m/api/hotel/getOffers",
        "https://www.expedia.com/api/hotel/offers"
    ]
    
    # Try each known endpoint first
    headers = generate_realistic_headers()
    async with httpx.AsyncClient(http2=True, timeout=30.0) as client:
        for endpoint in KNOWN_ENDPOINTS:
            try:
                await asyncio.sleep(random.uniform(1, 3))  # Random delay between attempts
                test_response = await client.get(endpoint, headers=headers)
                if test_response.status_code == 200:
                    return endpoint
            except Exception:
                continue
    
    # If all known endpoints fail, return the most likely one
    return "https://www.expedia.com/m/api/hotel/offers"

In [43]:
def generate_realistic_headers() -> Dict[str, str]:
    """Generate headers that better mimic a real browser session with proper typing"""
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1 Safari/605.1.15"
    ]
    
    return {
        "User-Agent": random.choice(user_agents),
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Referer": "https://www.expedia.com/",
        "Origin": "https://www.expedia.com",
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin",
        "Connection": "keep-alive",
        "TE": "Trailers",
        "x-expedia-client-id": "expedia.com",
        "x-expedia-client-name": "expedia.com-web-ui",
        "x-expedia-page-render-id": str(random.randint(1000000000, 9999999999)),
        "x-expedia-track-id": f"{random.randint(1000000000, 9999999999)}|{random.randint(1000000000, 9999999999)}",
        "x-requested-with": "XMLHttpRequest",
        "sec-ch-ua": '"Chromium";v="91", "Not;A Brand";v="99"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"'
    }

In [45]:
class RateLimiter:
    """Improved rate limiter with jitter and adaptive delays"""
    def __init__(self, max_requests_per_minute: int = 60):
        self.max_requests_per_minute = max_requests_per_minute
        self.request_times = []
        self.last_wait_time = 0
    
    async def wait(self) -> None:
        """Wait if we've made too many requests recently, with jitter"""
        now = time.time()
        
        # Remove old requests (older than 1 minute)
        self.request_times = [t for t in self.request_times if now - t < 60]
        
        if len(self.request_times) >= self.max_requests_per_minute:
            # Calculate how long to wait with jitter
            oldest_request = self.request_times[0]
            wait_time = max(0, 60 - (now - oldest_request) + random.uniform(0.1, 0.5))
            self.last_wait_time = wait_time
            if wait_time > 0:
                print(f"Rate limit approaching. Waiting {wait_time:.2f} seconds...")
                await asyncio.sleep(wait_time)
        
        self.request_times.append(time.time())
    
    def get_last_wait_time(self) -> float:
        """Get the last wait time for monitoring"""
        return self.last_wait_time

In [47]:
def parse_hotel_offer(response: Optional[Dict[str, Any]], hotel_code: str, check_in: str, check_out: str, adults: int) -> Optional[Dict[str, Any]]:
    """
    Robust parser that handles variations in Expedia's response structure
    with proper error handling and type hints
    """
    if not response:
        return None
    
    try:
        result = {
            "hotel_code": hotel_code,
            "check_in": check_in,
            "check_out": check_out,
            "adults": adults,
            "scrape_ts": datetime.utcnow().isoformat() + "Z",
            "total_price": None,
            "currency": None,
            "rate_plan": None,
            "cancellation_policy": None,
            "extras": []
        }
        
        # Safely navigate nested structures with defensive programming
        offers = response.get('offers', [])
        if not offers and 'rooms' in response:
            offers = response['rooms'][0].get('offers', []) if response.get('rooms') else []
        
        if offers and isinstance(offers, list) and len(offers) > 0:
            first_offer = offers[0]
            
            # Price info with nested checks
            price_info = first_offer.get('price', {})
            if isinstance(price_info, dict):
                total_price = price_info.get('total', {})
                if isinstance(total_price, dict):
                    result['total_price'] = total_price.get('amount')
                    result['currency'] = total_price.get('currency')
            
            # Rate plan
            if isinstance(first_offer.get('ratePlan'), dict):
                result['rate_plan'] = first_offer['ratePlan'].get('name')
            
            # Cancellation policy
            cancel_info = first_offer.get('cancellationInfo', {})
            if isinstance(cancel_info, dict):
                result['cancellation_policy'] = cancel_info.get('description')
            
            # Extras
            amenities = first_offer.get('amenities', [])
            if isinstance(amenities, list):
                result['extras'] = [
                    a.get('description') 
                    for a in amenities 
                    if isinstance(a, dict) and a.get('description')
                ]
        
        return result
    
    except Exception as e:
        print(f"Error parsing offer for hotel {hotel_code}: {str(e)[:200]}")
        return None

In [49]:
async def get_hotel_offers(
    hotel_code: str, 
    check_in: str, 
    check_out: str, 
    adults: int = 2, 
    max_retries: int = 3,
    client: Optional[httpx.AsyncClient] = None
) -> Optional[Dict[str, Any]]:
    """
    Improved hotel offers function with:
    - Better rate limiting
    - Cookie handling
    - Retry logic
    - Optional client reuse
    """
    url = "https://www.expedia.com/m/api/hotel/offers"
    params = {
        "hotelId": hotel_code,
        "checkIn": check_in,
        "checkOut": check_out,
        "adults": adults,
        "rooms": "1",
        "destination": hotel_code,
        "regionId": hotel_code,
    }
    
    headers = generate_realistic_headers()
    rate_limiter = RateLimiter(max_requests_per_minute=30)
    
    # Use provided client or create new one
    should_close = False
    if client is None:
        client = httpx.AsyncClient(http2=True, timeout=30.0)
        should_close = True
    
    try:
        for attempt in range(max_retries):
            try:
                await rate_limiter.wait()
                
                # Get fresh cookies on first attempt
                if attempt == 0:
                    await client.get("https://www.expedia.com/", headers=headers)
                
                response = await client.get(url, params=params, headers=headers)
                
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 429:
                    wait_time = (2 ** attempt) + random.random() * 2
                    print(f"Rate limited. Waiting {wait_time:.2f} seconds before retry...")
                    await asyncio.sleep(wait_time)
                    continue
                else:
                    response.raise_for_status()
                    
            except (httpx.RequestError, httpx.HTTPStatusError) as e:
                print(f"Attempt {attempt + 1} failed for hotel {hotel_code}: {str(e)[:200]}")
                if attempt == max_retries - 1:
                    return None
                await asyncio.sleep(1 + random.random())
    
    finally:
        if should_close:
            await client.aclose()
    
    return None

In [51]:
class ExpediaScraper:
    """Fixed scraper class with proper error handling and statistics"""
    def __init__(self, max_concurrent: int = 10, requests_per_second: int = 3):
        self.max_concurrent = max_concurrent
        self.requests_per_second = requests_per_second
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.request_counter = 0
        self.last_request_time = time.time()
        self.stats = {
            'total_requests': 0,
            'successful_requests': 0,
            'failed_requests': 0,
            'rate_limited': 0,
            'latencies': []
        }
        self.client = None
    
    async def __aenter__(self):
        self.client = httpx.AsyncClient(http2=True, timeout=30.0)
        return self
    
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.client:
            await self.client.aclose()
    
    async def rate_limit(self) -> None:
        """Ensure we don't exceed the rate limit"""
        self.request_counter += 1
        elapsed = time.time() - self.last_request_time
        
        if self.request_counter >= self.requests_per_second:
            if elapsed < 1.0:
                sleep_time = 1.0 - elapsed + random.uniform(0.1, 0.3)  # Add jitter
                await asyncio.sleep(sleep_time)
            self.request_counter = 0
            self.last_request_time = time.time()
    
    async def scrape_hotel(self, hotel_code: str, check_in: str, check_out: str, adults: int = 2) -> Optional[Dict[str, Any]]:
        """Scrape a single hotel with proper error handling"""
        async with self.semaphore:
            start_time = time.time()
            
            try:
                await self.rate_limit()
                
                response = await get_hotel_offers(
                    hotel_code, check_in, check_out, adults, 
                    client=self.client
                )
                parsed = parse_hotel_offer(response, hotel_code, check_in, check_out, adults)
                
                latency = time.time() - start_time
                self.stats['latencies'].append(latency)
                self.stats['total_requests'] += 1
                
                if parsed:
                    self.stats['successful_requests'] += 1
                    return parsed
                else:
                    self.stats['failed_requests'] += 1
                    return None
                    
            except Exception as e:
                self.stats['failed_requests'] += 1
                print(f"Error scraping hotel {hotel_code}: {str(e)[:200]}")
                return None
    
    async def scrape_batch(self, hotel_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Scrape a batch of hotels with proper typing"""
        tasks = []
        for data in hotel_data:
            task = asyncio.create_task(
                self.scrape_hotel(
                    data['hotel_code'], 
                    data['check_in'], 
                    data['check_out'], 
                    data.get('adults', 2)
                )
            )
            tasks.append(task)
        
        results = await asyncio.gather(*tasks)
        return [r for r in results if r is not None]
    
    def get_stats(self) -> Dict[str, Any]:
        """Get scraping statistics with division by zero protection"""
        stats = self.stats.copy()
        if stats['latencies']:
            stats['avg_latency'] = sum(stats['latencies']) / len(stats['latencies'])
            stats['p95_latency'] = sorted(stats['latencies'])[int(len(stats['latencies']) * 0.95)]
        else:
            stats['avg_latency'] = 0
            stats['p95_latency'] = 0
        
        # Calculate success rate safely
        total = stats['total_requests']
        if total > 0:
            stats['success_rate'] = (stats['successful_requests'] / total) * 100
        else:
            stats['success_rate'] = 0.0
            
        return stats

In [53]:
def read_input_csv(file_path: str) -> List[Dict[str, Any]]:
    """Read input CSV file with error handling"""
    try:
        df = pd.read_csv(file_path)
        required_columns = {'hotel_code', 'check_in', 'check_out'}
        if not required_columns.issubset(df.columns):
            raise ValueError(f"CSV missing required columns: {required_columns}")
        return df.to_dict('records')
    except Exception as e:
        print(f"Error reading input CSV: {e}")
        return []

def write_output_ndjson(data: List[Dict[str, Any]], file_path: str) -> bool:
    """Write output to NDJSON file with error handling"""
    try:
        with open(file_path, 'w', encoding='utf-8') as f:
            for item in data:
                if item:  # Skip None values
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
        return True
    except Exception as e:
        print(f"Error writing output NDJSON: {e}")
        return False

In [55]:
async def main(
    input_file: str, 
    output_file: str, 
    max_concurrent: int = 10, 
    requests_per_second: int = 3,
    batch_size: int = 100
) -> bool:
    """Main scraping function with comprehensive error handling"""
    print(f"Starting scrape from {input_file} to {output_file}")
    start_time = time.time()
    
    # Read input
    hotel_data = read_input_csv(input_file)
    if not hotel_data:
        print("No valid input data found")
        return False
    
    print(f"Loaded {len(hotel_data)} hotel queries")
    
    # Initialize scraper with context manager
    async with ExpediaScraper(max_concurrent, requests_per_second) as scraper:
        all_results = []
        
        # Process in batches to manage memory
        for i in range(0, len(hotel_data), batch_size):
            batch = hotel_data[i:i + batch_size]
            batch_num = (i // batch_size) + 1
            total_batches = (len(hotel_data) - 1) // batch_size + 1
            
            print(f"\nProcessing batch {batch_num}/{total_batches}")
            
            results = await scraper.scrape_batch(batch)
            all_results.extend(results)
            
            # Write intermediate results
            temp_file = f"temp_output_{i}.ndjson"
            write_output_ndjson(results, temp_file)
            
            # Print stats
            stats = scraper.get_stats()
            print(f"Progress: {stats['successful_requests']}/{stats['total_requests']} successful")
            print(f"Success rate: {stats['success_rate']:.2f}%")
            print(f"Avg latency: {stats['avg_latency']:.2f}s, P95 latency: {stats['p95_latency']:.2f}s")
        
        # Write final output
        write_success = write_output_ndjson(all_results, output_file)
        
        # Final stats
        total_time = time.time() - start_time
        stats = scraper.get_stats()
        
        print("\nScraping complete!")
        print(f"Total time: {total_time:.2f} seconds")
        print(f"Total requests: {stats['total_requests']}")
        print(f"Successful requests: {stats['successful_requests']}")
        print(f"Success rate: {stats['success_rate']:.2f}%")
        if total_time > 0:
            print(f"Requests per second: {stats['total_requests']/total_time:.2f}")
        print(f"Avg latency: {stats['avg_latency']:.2f}s")
        print(f"P95 latency: {stats['p95_latency']:.2f}s")
        
        return write_success and stats['success_rate'] > 0

In [57]:

sample_input = [
    {"hotel_code": "276006", "check_in": "2025-11-12", "check_out": "2025-11-14", "adults": 2},
    {"hotel_code": "12345", "check_in": "2025-11-15", "check_out": "2025-11-17", "adults": 2},
    {"hotel_code": "67890", "check_in": "2025-11-18", "check_out": "2025-11-20", "adults": 2},
]

pd.DataFrame(sample_input).to_csv('input_sample.csv', index=False)

# Run the main function
await main('input_sample.csv', 'output_sample.ndjson', max_concurrent=3, requests_per_second=2)

Starting scrape from input_sample.csv to output_sample.ndjson
Loaded 3 hotel queries

Processing batch 1/1
Attempt 1 failed for hotel 276006: <StreamReset stream_id:1, error_code:1, remote_reset:True>
Attempt 1 failed for hotel 12345: <StreamReset stream_id:3, error_code:1, remote_reset:True>
Attempt 1 failed for hotel 67890: <StreamReset stream_id:5, error_code:1, remote_reset:True>
Attempt 2 failed for hotel 276006: <StreamReset stream_id:7, error_code:1, remote_reset:True>
Attempt 2 failed for hotel 67890: <StreamReset stream_id:9, error_code:1, remote_reset:True>
Attempt 2 failed for hotel 12345: <StreamReset stream_id:11, error_code:1, remote_reset:True>
Attempt 3 failed for hotel 67890: <StreamReset stream_id:13, error_code:1, remote_reset:True>
Attempt 3 failed for hotel 276006: <StreamReset stream_id:15, error_code:1, remote_reset:True>
Attempt 3 failed for hotel 12345: <StreamReset stream_id:17, error_code:1, remote_reset:True>
Progress: 0/3 successful
Success rate: 0.00%
Avg 

False

# SECOND SOLUTION 

# bUT NEED EXPEDIA API KEYS FOR ACCURATE INFORMATION & BEST SOLUTIONS 

In [73]:
import asyncio
import httpx
import json
import pandas as pd
from datetime import datetime
import random
import time
import nest_asyncio
from bs4 import BeautifulSoup
import os
from dotenv import load_dotenv
from typing import Optional, Dict, Any, List

# Enable nested asyncio for Jupyter
nest_asyncio.apply()

# Load environment variables if any
load_dotenv()

class ExpediaScraper:
    def __init__(self, max_concurrent: int = 3, requests_per_second: int = 2):
        self.max_concurrent = max_concurrent
        self.requests_per_second = requests_per_second
        self.stats = {
            'total_requests': 0,
            'successful_requests': 0,
            'latencies': [],
            'start_time': time.time()
        }
        
        # Realistic headers
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'application/json',
            'Accept-Language': 'en-US,en;q=0.9',
            'Referer': 'https://www.expedia.com/',
            'Origin': 'https://www.expedia.com',
            'Sec-Fetch-Dest': 'empty',
            'Sec-Fetch-Mode': 'cors',
            'Sec-Fetch-Site': 'same-origin',
        }

    async def scrape_hotel(self, hotel_code: str, check_in: str, check_out: str, adults: int = 2) -> Optional[Dict[str, Any]]:
        """Scrape a single hotel's data from Expedia"""
        url = f"https://www.expedia.com/api/hotel/offers?hotelId={hotel_code}&checkIn={check_in}&checkOut={check_out}&adults={adults}"
        
        async with httpx.AsyncClient(headers=self.headers, timeout=30.0) as client:
            for attempt in range(3):  # Retry up to 3 times
                try:
                    start_time = time.time()
                    response = await client.get(url)
                    self.stats['total_requests'] += 1
                    
                    if response.status_code == 200:
                        data = response.json()
                        self.stats['successful_requests'] += 1
                        self.stats['latencies'].append(time.time() - start_time)
                        return {
                            'hotel_code': hotel_code,
                            'check_in': check_in,
                            'check_out': check_out,
                            'adults': adults,
                            'data': data,
                            'timestamp': datetime.now().isoformat(),
                            'status': 'success'
                        }
                    elif response.status_code == 429:
                        retry_after = int(response.headers.get('Retry-After', 10))
                        print(f"Rate limited. Waiting {retry_after} seconds...")
                        await asyncio.sleep(retry_after)
                        continue
                    else:
                        print(f"Attempt {attempt + 1} failed for hotel {hotel_code}: HTTP {response.status_code}")
                except Exception as e:
                    print(f"Attempt {attempt + 1} failed for hotel {hotel_code}: {str(e)}")
                
                if attempt < 2:  # Don't sleep after last attempt
                    await asyncio.sleep(random.uniform(1, 3))
            
            return {
                'hotel_code': hotel_code,
                'check_in': check_in,
                'check_out': check_out,
                'adults': adults,
                'data': None,
                'timestamp': datetime.now().isoformat(),
                'status': 'failed',
                'error': 'Max retries exceeded'
            }

    async def scrape_batch(self, hotel_queries: List[Dict[str, Any]]) -> List[Optional[Dict[str, Any]]]:
        """Scrape multiple hotels with rate limiting"""
        semaphore = asyncio.Semaphore(self.max_concurrent)
        
        async def limited_task(hotel_data):
            async with semaphore:
                # Rate limiting
                await asyncio.sleep(1 / self.requests_per_second)
                return await self.scrape_hotel(
                    hotel_data['hotel_code'],
                    hotel_data['check_in'],
                    hotel_data['check_out'],
                    hotel_data.get('adults', 2)
                )
        
        tasks = [limited_task(data) for data in hotel_queries]
        return await asyncio.gather(*tasks)
    
    def get_stats(self):
        """Get scraping statistics"""
        stats = self.stats.copy()
        stats['total_time'] = time.time() - stats['start_time']
        
        if stats['latencies']:
            stats['avg_latency'] = sum(stats['latencies']) / len(stats['latencies'])
            stats['p95_latency'] = sorted(stats['latencies'])[int(len(stats['latencies']) * 0.95)]
        else:
            stats['avg_latency'] = 0
            stats['p95_latency'] = 0
        
        stats['success_rate'] = (
            (stats['successful_requests'] / stats['total_requests'] * 100) 
            if stats['total_requests'] > 0 else 0
        )
        
        stats['requests_per_second'] = (
            stats['total_requests'] / stats['total_time'] 
            if stats['total_time'] > 0 else 0
        )
        
        return stats

async def main(input_file: str, output_file: str, max_concurrent: int = 3, requests_per_second: int = 2):
    """Main function to run the scraper"""
    print(f"Starting scrape from {input_file} to {output_file}")
    
    # Read input
    df = pd.read_csv(input_file)
    hotel_queries = df.to_dict('records')
    print(f"Loaded {len(hotel_queries)} hotel queries\n")
    
    # Initialize scraper
    scraper = ExpediaScraper(max_concurrent=max_concurrent, requests_per_second=requests_per_second)
    
    # Process in batches if needed
    batch_size = max_concurrent * 5
    results = []
    
    for i in range(0, len(hotel_queries), batch_size):
        batch = hotel_queries[i:i + batch_size]
        print(f"Processing batch {i // batch_size + 1}/{(len(hotel_queries) - 1) // batch_size + 1}")
        
        batch_results = await scraper.scrape_batch(batch)
        results.extend(batch_results)
        
        # Save progress after each batch
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in results:
                if item:
                    f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        # Print progress
        stats = scraper.get_stats()
        print(f"Progress: {stats['successful_requests']}/{stats['total_requests']} successful")
        print(f"Success rate: {stats['success_rate']:.2f}%")
        print(f"Avg latency: {stats['avg_latency']:.2f}s, P95 latency: {stats['p95_latency']:.2f}s\n")
    
    # Final stats
    stats = scraper.get_stats()
    print("\nScraping complete!")
    print(f"Total time: {stats['total_time']:.2f} seconds")
    print(f"Total requests: {stats['total_requests']}")
    print(f"Successful requests: {stats['successful_requests']}")
    print(f"Success rate: {stats['success_rate']:.2f}%")
    print(f"Requests per second: {stats['requests_per_second']:.2f}")
    print(f"Avg latency: {stats['avg_latency']:.2f}s")
    print(f"P95 latency: {stats['p95_latency']:.2f}s")

# Create sample input
sample_input = [
    {"hotel_code": "276006", "check_in": "2025-11-12", "check_out": "2025-11-14", "adults": 2},
    {"hotel_code": "12345", "check_in": "2025-11-15", "check_out": "2025-11-17", "adults": 2},
    {"hotel_code": "67890", "check_in": "2025-11-18", "check_out": "2025-11-20", "adults": 2},
]

pd.DataFrame(sample_input).to_csv('input_sample.csv', index=False)

# Run the main function
await main('input_sample.csv', 'output_sample.ndjson', max_concurrent=3, requests_per_second=2)

Starting scrape from input_sample.csv to output_sample.ndjson
Loaded 3 hotel queries

Processing batch 1/1
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Rate limited. Waiting 10 seconds...
Progress: 0/9 successful
Success rate: 0.00%
Avg latency: 0.00s, P95 latency: 0.00s


Scraping complete!
Total time: 40.22 seconds
Total requests: 9
Successful requests: 0
Success rate: 0.00%
Requests per second: 0.22
Avg latency: 0.00s
P95 latency: 0.00s


In [ ]:
This is example script 

In [1]:
import requests
import time

# Configuration
API_KEY = "Need ExpediaAPI keys"  # From developer portal
HOTEL_ID = "381807"  # Example: The Ritz-Carlton, SF
CHECK_IN = "2024-12-01"
CHECK_OUT = "2024-12-05"

def get_hotel_offers():
    url = "https://test.api.expedia.com/partner/v3/hotel-offers"
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/vnd.exp-hotel.v3+json"
    }

    params = {
        "hotelId": HOTEL_ID,
        "checkInDate": CHECK_IN,
        "checkOutDate": CHECK_OUT,
        "adults": 2,
        "roomQuantity": 1,
        "currency": "USD"
    }

    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            print(f"Success! Found offers for: {data['hotel']['name']}")
            return data
        elif response.status_code == 401:
            print("Error: Invalid API Key")
        elif response.status_code == 404:
            print("Error: Hotel not found")
        else:
            print(f"Error {response.status_code}: {response.text[:200]}")
            
    except Exception as e:
        print(f"Request failed: {str(e)}")
        return None

# Execute
result = get_hotel_offers()

# Example of parsing response
if result:
    print("\nSample Data:")
    print(f"Hotel ID: {result['hotel']['hotelId']}")
    print(f"Check-in: {result['offers'][0]['checkInDate']}")
    print(f"Price: {result['offers'][0]['price']['total']} USD")

Request failed: HTTPSConnectionPool(host='test.api.expedia.com', port=443): Max retries exceeded with url: /partner/v3/hotel-offers?hotelId=381807&checkInDate=2024-12-01&checkOutDate=2024-12-05&adults=2&roomQuantity=1&currency=USD (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001D72F4E5A00>: Failed to resolve 'test.api.expedia.com' ([Errno 11001] getaddrinfo failed)"))
